# LoRA Q4 - Proper Block-wise Quantization

Uses bitsandbytes actual quantization with per-block absmax.

**Key insight**: Q4 stores:
- Packed uint8 indices (2 x 4-bit per byte)
- Block absmax values (scales)

We train LoRA to reproduce the quantized floats directly.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/LoRA_Compress'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/databases', exist_ok=True)

In [ ]:
%cd /content
!git clone https://github.com/qades/loracompress.git 2>/dev/null || true
%cd loracompress
!pip install -q transformers torch optuna tqdm bitsandbytes accelerate

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import numpy as np
from transformers import AutoModelForCausalLM

# Import bitsandbytes quantization functions
import bitsandbytes as bnb
from bitsandbytes.functional import quantize_blockwise, dequantize_blockwise

print(f'PyTorch: {torch.__version__}')
print(f'bitsandbytes: {bnb.__version__}')

## Step 4: Load FP16 and Quantize Properly

In [ ]:
MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'

print('Loading FP16 model...')
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map='cpu',
    trust_remote_code=True,
)
print('FP16 model loaded\n')

def quantize_layer_bnb(weight, blocksize=64):
    """
    Proper block-wise NF4 quantization using bitsandbytes.
    Returns quantized representation that can be dequantized.
    """
    w = weight.float().cuda()  # bnb functions need CUDA
    
    # Quantize using bnb - this creates proper block-wise quantization
    # Returns (quantized_weights, quant_state)
    w_quantized, quant_state = bnb.functional.quantize_nf4(w, blocksize=blocksize)
    
    # Dequantize to get the actual Q4 values
    w_dequantized = bnb.functional.dequantize_nf4(w_quantized, quant_state)
    
    # Extract info from quant_state
    # quant_state = (absmax, shape, dtype, blocksize, nested_quant_state)
    absmax = quant_state[0] if isinstance(quant_state, tuple) else quant_state.absmax
    
    return {
        'fp16': w.cpu(),
        'q4_packed': w_quantized.cpu(),
        'q4_dequantized': w_dequantized.cpu(),
        'absmax': absmax.cpu() if torch.is_tensor(absmax) else None,
        'quant_state': quant_state,
        'blocksize': blocksize,
    }

print('Quantization function defined')

## Step 5: Analyze Layers - Q4 vs FP16

In [ ]:
print('Analyzing layers with proper Q4 quantization\n')
print('=' * 80)

TARGET_LAYERS = [
    'model.layers.0.self_attn.q_proj',
    'model.layers.15.self_attn.q_proj',
    'model.layers.15.mlp.up_proj',
]

layer_data = []

for target in TARGET_LAYERS:
    for name, param in model_fp16.named_parameters():
        if target in name and name.endswith('.weight'):
            print(f'\n📊 {name}')
            print(f'   Shape: {tuple(param.shape)}')
            
            # Quantize
            qdata = quantize_layer_bnb(param.data, blocksize=64)
            
            # Stats
            w_fp16 = qdata['fp16']
            w_q4 = qdata['q4_dequantized']
            
            print(f'\n   FP16: mean={w_fp16.mean():.4f}, std={w_fp16.std():.4f}')
            print(f'   Q4:   mean={w_q4.mean():.4f}, std={w_q4.std():.4f}')
            
            # Quantization error
            l1_err = torch.mean(torch.abs(w_q4 - w_fp16)).item() / torch.mean(torch.abs(w_fp16)).item() * 100
            print(f'\n   Quantization error: {l1_err:.2f}% L1')
            
            # SVD comparison
            def svd_stats(w, ranks=[16, 32, 64]):
                try:
                    U, S, Vh = torch.linalg.svd(w, full_matrices=False)
                    total = (S ** 2).sum()
                    if total < 1e-10:
                        return {r: 0.0 for r in ranks}
                    return {r: ((S[:r] ** 2).sum() / total * 100).item() for r in ranks if r <= len(S)}
                except:
                    return {r: 0.0 for r in ranks}
            
            stats_fp16 = svd_stats(w_fp16)
            stats_q4 = svd_stats(w_q4)
            
            print(f'\n   SVD Variance Explained:')
            for r in [16, 32, 64]:
                if r in stats_q4:
                    diff = stats_q4[r] - stats_fp16[r]
                    print(f'     Rank {r:2d}: Q4={stats_q4[r]:5.1f}% FP16={stats_fp16[r]:5.1f}% ({diff:+5.1f}%)')
            
            layer_data.append({
                'name': name,
                'fp16': w_fp16,
                'q4_dequantized': w_q4,
                'q4_packed': qdata['q4_packed'],
                'stats_fp16': stats_fp16,
                'stats_q4': stats_q4,
            })
            break

print('\n' + '=' * 80)

## Step 6: Train LoRA on Q4 Dequantized Weights

In [ ]:
# Select layer
idx = 1
selected = layer_data[idx]
print(f"Training on: {selected['name']}")

# Target is Q4 dequantized weights
target_weight = selected['q4_dequantized']
target_fp16 = selected['fp16']
print(f'Target shape: {target_weight.shape}')
print(f'Q4 vs FP16 SVD diff at rank 32: {selected["stats_q4"][32] - selected["stats_fp16"][32]:.1f}%')

In [ ]:
# Training functions
def train_lora(target, rank, lr, epochs, device='cuda'):
    d, k = target.shape
    target_d = target.float().to(device)
    
    A = nn.Parameter(torch.randn(rank, k, device=device) * 0.01)
    B = nn.Parameter(torch.randn(d, rank, device=device) * 0.01)
    opt = torch.optim.AdamW([A, B], lr=lr)
    
    best_loss, best_A, best_B = float('inf'), None, None
    
    for epoch in range(epochs):
        opt.zero_grad()
        W = torch.matmul(B, A)
        loss = F.mse_loss(W, target_d)
        if not torch.isfinite(loss):
            return float('inf'), 0
        loss.backward()
        opt.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_A, best_B = A.detach().clone(), B.detach().clone()
    
    with torch.no_grad():
        W_best = torch.matmul(best_B, best_A)
        l1 = torch.mean(torch.abs(W_best - target_d)).item() / torch.mean(torch.abs(target_d)).item() * 100
    return l1, epoch + 1

# Quick test
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nQuick test on {device}:\n')

for rank in [16, 32, 64]:
    err_q4, _ = train_lora(target_weight, rank, 0.01, 300, device)
    err_fp16, _ = train_lora(target_fp16, rank, 0.01, 300, device)
    print(f'Rank {rank:2d}: Q4 target={err_q4:.2f}% | FP16 target={err_fp16:.2f}%')

In [ ]:
# Optuna optimization
def objective(trial):
    rank = trial.suggest_int('rank', 8, 128, log=True)
    lr = trial.suggest_float('lr', 0.001, 0.1, log=True)
    epochs = trial.suggest_int('epochs', 200, 1000, log=True)
    
    l1_err, actual_epochs = train_lora(target_weight, rank, lr, epochs, device)
    
    trial.set_user_attr('l1_error', l1_err)
    trial.set_user_attr('actual_epochs', actual_epochs)
    
    d, k = target_weight.shape
    compression = (d * k) / (rank * (d + k))
    
    if l1_err <= 5.0:
        return -compression * 10
    return 1000 + l1_err * 10

import os
study_name = f'proper_q4_{selected["name"].replace(".", "_")}'
db_file = f'{DRIVE_BASE}/databases/{study_name}.db'
os.makedirs(os.path.dirname(db_file), exist_ok=True)

study = optuna.create_study(
    study_name=study_name,
    storage=f'sqlite:///{db_file}',
    direction='minimize',
    load_if_exists=True
)

print(f'Running 30 trials...\n')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\nBest: L1={study.best_trial.user_attrs["l1_error"]:.2f}%')
print(f'Rank: {study.best_params["rank"]}, LR: {study.best_params["lr"]:.4f}')